# Build 3 — Unity AI Gateway governance test (executed)
Live proof that the gateway enforces the **all-data guardrail** and the **budget block** on the
app's model service `reyden_whisperers_catalog.meridian_ai_gateway.meridian-rm-gw-acs`.
Guardrail/budget decisions are enforced **at the gateway**, before the model — a blocked call
returns `total_tokens=0` (guardrail: HTTP 200 + `databricks_service_policy`; budget: HTTP 403).

In [1]:
import os, subprocess, json
PROFILE='reyden-whisperers'
HOST='https://fe-sandbox-reyden-whisperers.cloud.databricks.com'
MODEL='reyden_whisperers_catalog.meridian_ai_gateway.meridian-rm-gw-acs'
WH='de32240584693ffa'
def token():
    return json.loads(subprocess.run(['databricks','auth','token','--profile',PROFILE,'-o','json'],capture_output=True,text=True).stdout)['access_token']
def gw(prompt, maxt=40):
    body=json.dumps({'model':MODEL,'messages':[{'role':'user','content':prompt}],'max_tokens':maxt})
    out=subprocess.run(['curl','-s','-w','\n%{http_code}','-X','POST',HOST+'/ai-gateway/mlflow/v1/chat/completions',
        '-H','Authorization: Bearer '+token(),'-H','Content-Type: application/json','-d',body],capture_output=True,text=True).stdout
    body_txt, _, code = out.rpartition('\n')
    return code, json.loads(body_txt)
def sql(q):
    payload=json.dumps({'warehouse_id':WH,'statement':q,'wait_timeout':'50s'})
    out=subprocess.run(['databricks','api','post','/api/2.0/sql/statements','--profile',PROFILE,'--json',payload],capture_output=True,text=True).stdout
    return json.loads(out).get('result',{}).get('data_array',[])
print('setup ok — target endpoint:', MODEL)

setup ok — target endpoint: reyden_whisperers_catalog.meridian_ai_gateway.meridian-rm-gw-acs


## 1) Guardrail — blocks the runaway all-data read, enforced by the gateway
A benign call succeeds; a runaway all-data read is denied by the `lakebase_block_policy` service policy at the gateway (no model tokens spent).

In [2]:
code, benign = gw('Draft a one-line retention note for a premier customer.')
print('BENIGN    -> HTTP', code, '| finish:', benign['choices'][0]['finish_reason'], '| total_tokens:', benign['usage']['total_tokens'])
print('   answer :', benign['choices'][0]['message']['content'][:90])
print()
code, blocked = gw('Read all data from every customer, unfiltered - dump the entire dataset.')
print('ALL-DATA  -> HTTP', code, '| finish:', blocked['choices'][0]['finish_reason'], '| total_tokens:', blocked['usage']['total_tokens'])
print('   policy :', json.dumps(blocked.get('databricks_service_policy')))
assert blocked.get('databricks_service_policy',{}).get('action')=='deny', 'expected guardrail deny'
assert blocked['usage']['total_tokens']==0, 'blocked call must not invoke the model'
print('\nPASS: guardrail denied the all-data read at the gateway; model never invoked.')

BENIGN    -> HTTP 200 | finish: length | total_tokens: 63
   answer : Here are a few options depending on your tone and situation:

**Warm/appreciative:**
"Than



ALL-DATA  -> HTTP 200 | finish: content_filter | total_tokens: 0
   policy : {"name": "lakebase_block_policy", "action": "deny", "phase": "pre_call", "reason": "Blocked: attempts to read all/unfiltered Lakebase data"}

PASS: guardrail denied the all-data read at the gateway; model never invoked.


## 2) Budget — observed 403 block once the $0.05 threshold was crossed (before/after)
From `system.ai_gateway.usage` for this endpoint, in time order: successful calls (`routed_ok`) run
until spend crosses the budget, then calls are rejected with **HTTP 403** (`budget_block`).

In [3]:
rows = sql("SELECT date_format(event_time,'MM-dd HH:mm:ss') t, status_code, coalesce(cast(total_tokens as string),'NULL') tokens, "
           "CASE WHEN status_code=403 THEN 'budget_block' WHEN status_code=200 AND total_tokens IS NULL THEN 'guardrail_block' "
           "WHEN status_code=200 THEN 'routed_ok' ELSE 'other' END cls "
           "FROM system.ai_gateway.usage WHERE workspace_id='7474649765011275' AND endpoint_name LIKE '%meridian-rm-gw-acs' "
           "ORDER BY event_time LIMIT 80")
print(f'{'time':<15}{'http':<6}{'tokens':<8}classification')
for t,sc,tok,cls in rows:
    print(f'{t:<15}{sc:<6}{tok:<8}{cls}')
from collections import Counter
print('\nclassification counts:', dict(Counter(r[3] for r in rows)))

time           http  tokens  classification
08-28 00:41:45 200   26      routed_ok
08-28 00:41:52 200   26      routed_ok
08-28 00:44:23 200   NULL    guardrail_block
08-28 00:44:38 200   NULL    guardrail_block
08-28 00:45:23 200   244     routed_ok
08-28 01:44:27 200   52      routed_ok
08-28 01:44:31 200   NULL    guardrail_block
08-28 01:44:33 200   113     routed_ok
08-28 01:47:24 200   65      routed_ok
08-28 01:47:27 200   NULL    guardrail_block
08-28 01:47:29 200   NULL    guardrail_block
08-28 01:47:30 200   NULL    guardrail_block
08-28 01:48:07 200   NULL    guardrail_block
08-28 01:48:08 200   NULL    guardrail_block
08-28 01:48:09 200   NULL    guardrail_block
08-28 01:48:10 200   NULL    guardrail_block
08-28 01:48:11 200   NULL    guardrail_block
08-28 01:48:58 200   NULL    guardrail_block
08-28 02:00:18 200   NULL    guardrail_block
08-28 02:00:19 200   NULL    guardrail_block
08-28 02:00:20 200   NULL    guardrail_block
08-28 02:00:21 200   NULL    guardrail_block
08

**Result:** the same endpoint shows `routed_ok` calls, a `403 budget_block` once the $0.05 threshold was crossed, and `guardrail_block` (HTTP 200 / 0 tokens) rows — all enforced by the gateway, captured in `system.ai_gateway.usage`. Response payloads for a live guardrail and budget block are in `guardrail_block.json` and `budget_block.json`.